# DS2002 · ACC Reality Check

**Team MNM · Fall 2026**

---

## ACC Reality Check: team notebook

Which ACC men's basketball teams are as good as their records, and which have built their records on easy opponents?

Work in progress: the team setup and first source pulls are complete. Transformations, SQLite loading, SQL answers, charts, and the published API are still to come. The API code will live in `api/main.py`.

### 0. Team, sources, and setup

**Team:** MNM  
**Members:** Nathaniel Garcia, Mouad Tazka, Mohamed Hassan  
**Repository:** https://github.com/NateG-10/ds-2002-final-project  
**Meetings:** As needed, but at least once a week

| Member | Responsibility | Questions |
|---|---|---|
| Nathaniel Garcia | Ratings CSV and crosswalk | Q1 and Q2 |
| Mouad Tazka | ESPN pulls and box scores | Q4 and Q5 |
| Mohamed Hassan | SQLite schema and queries; API and PythonAnywhere | Q3 and Q6 |

**Sources and current coverage**
- Torvik: https://barttorvik.com/2027_team_results.csv — snapshot downloaded November 18, 2026.
- ESPN scoreboard: https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard — downloaded November 18, 2026, for games on November 14 (`groups=2`, `dates=20261114`, `limit=200`).
- ESPN summary endpoint (not pulled yet): https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/summary

Run from the repository root with a Python kernel containing `pandas` and `requests`. If needed, run `%pip install pandas requests` in a temporary notebook cell, then restart the kernel. Run cells from top to bottom. Saved raw files are reused; current outputs are exploratory, not final-project results.

In [1]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import csv
import json
import requests
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'team-notebook.ipynb').exists():
    raise RuntimeError('Run this notebook with the repository root as the working directory.')
RAW = ROOT / 'raw'
RAW.mkdir(exist_ok=True)
DB_PATH = ROOT / 'acc.db'
PULL_DATE = '2026-11-18'
GAME_DATE = '20261114'
TORVIK_URL = 'https://barttorvik.com/2027_team_results.csv'
ESPN_URL = 'https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard'
SUMMARY_URL = 'https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/summary'

def saved_response(url, path, params=None, snapshot_date=None):
    if path.exists():
        print('Reading saved response:', path.name)
    else:
        today = datetime.now(ZoneInfo('America/New_York')).date().isoformat()
        if snapshot_date is not None and snapshot_date != today:
            raise ValueError('Cannot recreate an old Torvik snapshot from a live URL. '
                             'Restore the saved file or choose today as PULL_DATE.')
        response = requests.get(url, params=params, timeout=45)
        response.raise_for_status()
        if 'json' in path.suffix:
            response.json()  # Reject a non-JSON response before caching it.
        elif response.text.lstrip().startswith('<'):
            raise ValueError('Expected CSV, received an HTML page.')
        path.write_bytes(response.content)
        metadata = {'url': response.url, 'status_code': response.status_code,
                    'content_type': response.headers.get('content-type'),
                    'pulled_at_eastern': datetime.now(ZoneInfo('America/New_York')).isoformat(),
                    'body_file': path.name}
        path.with_name(path.stem + '_metadata.json').write_text(json.dumps(metadata, indent=2))
        print('Downloaded:', response.url)
    return path


### 1. Extract

The saved kickoff responses were copied unchanged into the root `raw/` folder. Reruns read those files rather than call the sources again.

Current coverage: one Torvik snapshot and the November 14 scoreboard. **Still to do:** scoreboards for every date from November 1 through the most recent finished games, summaries for finished games, and new ratings snapshots at least twice a week. A later live download cannot recreate a missing November 18 ratings snapshot.

In [2]:
ratings_path = saved_response(TORVIK_URL, RAW / f'torvik_{PULL_DATE}.csv', snapshot_date=PULL_DATE)
ratings = pd.read_csv(ratings_path)
print('Shape:', ratings.shape)
print('Columns:', ratings.columns.tolist())
display(ratings.head())
acc = ratings.loc[ratings['conf'].eq('ACC')]
print('ACC rows:', len(acc))
display(acc[['team', 'conf', 'record', 'rank', 'rank.1', 'barthag']])
with ratings_path.open(newline='') as f:
    original_headers = next(csv.reader(f))
print('Original headers named rank:', original_headers.count('rank'))
print('Rank columns equal in this snapshot:', ratings['rank'].equals(ratings['rank.1']))
print('Barthag decreases as rank.1 increases:',
      ratings.sort_values('rank.1')['barthag'].is_monotonic_decreasing)


Reading saved response: torvik_2026-11-18.csv
Shape: (365, 45)
Columns: ['rank', 'team', 'conf', 'record', 'adjoe', 'oe Rank', 'adjde', 'de Rank', 'barthag', 'rank.1', 'proj. W', 'Proj. L', 'Pro Con W', 'Pro Con L', 'Con Rec.', 'sos', 'ncsos', 'consos', 'Proj. SOS', 'Proj. Noncon SOS', 'Proj. Con SOS', 'elite SOS', 'elite noncon SOS', 'Opp OE', 'Opp DE', 'Opp Proj. OE', 'Opp Proj DE', 'Con Adj OE', 'Con Adj DE', 'Qual O', 'Qual D', 'Qual Barthag', 'Qual Games', 'FUN', 'ConPF', 'ConPA', 'ConPoss', 'ConOE', 'ConDE', 'ConSOSRemain', 'Conf Win%', 'WAB', 'WAB Rk', 'Fun Rk', 'adjt']


,rank,team,conf,record,adjoe,oe Rank,adjde,de Rank,barthag,rank.1,...,ConPA,ConPoss,ConOE,ConDE,ConSOSRemain,Conf Win%,WAB,WAB Rk,Fun Rk,adjt
0,1,Kansas,B12,5-0,122.210929,6,92.797460,3,0.959546,1,...,0.0,0.0,0,0,0.843819,0,1.603877,2,80,72.550470
1,2,Houston,B12,5-0,122.416439,5,93.279053,5,0.957956,2,...,0.0,0.0,0,0,0.847790,0,1.592145,3,59,67.258776
2,3,Duke,ACC,3-2,119.880679,14,92.060397,1,0.954200,3,...,0.0,0.0,0,0,0.805033,0,-0.442862,120,353,71.811124
3,4,Illinois,B10,4-1,124.777422,1,96.411380,19,0.951011,4,...,0.0,0.0,0,0,0.839177,0,0.981213,10,116,74.254181
4,5,Florida,SEC,5-1,123.133092,2,95.403194,11,0.949515,5,...,0.0,0.0,0,0,0.858506,0,1.732562,1,100,69.545494


ACC rows: 18


,team,conf,record,rank,rank.1,barthag
2,Duke,ACC,3-2,3,3,0.954200
5,Virginia,ACC,6-0,6,6,0.944878
15,Miami FL,ACC,4-1,16,16,0.920269
17,Louisville,ACC,2-1,18,18,0.919139
31,N.C. State,ACC,4-0,32,32,0.881941
36,Virginia Tech,ACC,5-0,37,37,0.873199
48,North Carolina,ACC,3-1,49,49,0.847129
52,Florida St.,ACC,4-1,53,53,0.834588
60,SMU,ACC,5-0,61,61,0.822848
67,Notre Dame,ACC,5-1,68,68,0.811654


Original headers named rank: 2
Rank columns equal in this snapshot: True
Barthag decreases as rank.1 increases: True


In [3]:
scoreboard_path = saved_response(
    ESPN_URL, RAW / f'espn_scoreboard_{GAME_DATE}.json',
    params={'groups': 2, 'dates': GAME_DATE, 'limit': 200})
scoreboard = json.loads(scoreboard_path.read_text())
print('Top-level keys:', list(scoreboard))
events = scoreboard['events']
print('Games:', len(events))
if events:
    print('First event keys:', list(events[0]))
    print('First competition keys:', list(events[0]['competitions'][0]))
for event in events:
    competition = event['competitions'][0]
    print('\nGame ID:', event['id'])
    print('Start exactly as ESPN wrote it:', competition['date'])
    print('Eastern time:', pd.Timestamp(competition['date']).tz_convert('America/New_York'))
    print('Finished:', competition['status']['type']['completed'])
    print('Neutral site:', competition['neutralSite'])
    for side in competition['competitors']:
        print(side['homeAway'], side['team']['location'],
              '| full name:', side['team']['displayName'],
              '| score:', repr(side['score']), '| type:', type(side['score']).__name__)



Reading saved response: espn_scoreboard_20261114.json
Top-level keys: ['leagues', 'groups', 'events', 'provider', 'eventsDate']
Games: 3
First event keys: ['id', 'uid', 'date', 'name', 'shortName', 'season', 'competitions', 'links', 'status']
First competition keys: ['id', 'uid', 'date', 'attendance', 'type', 'timeValid', 'neutralSite', 'conferenceCompetition', 'playByPlayAvailable', 'recent', 'venue', 'competitors', 'notes', 'status', 'broadcasts', 'format', 'startDate', 'broadcast', 'geoBroadcasts', 'highlights', 'headlines']

Game ID: 401920307
Start exactly as ESPN wrote it: 2026-11-14T21:00Z
Eastern time: 2026-11-14 16:00:00-05:00
Finished: True
Neutral site: False
home Georgia Tech | full name: Georgia Tech Yellow Jackets | score: '75' | type: str
away UNC Asheville | full name: UNC Asheville Bulldogs | score: '74' | type: str

Game ID: 401909728
Start exactly as ESPN wrote it: 2026-11-14T21:00Z
Eastern time: 2026-11-14 16:00:00-05:00
Finished: True
Neutral site: False
home Wake 

In [4]:
# TODO: ESPN game summaries, one request per finished game

### 2. Transform

**TODO:** one function or clearly labeled cell per step, each printing a row count.

Kickoff observations to carry into this work: 365 Torvik teams, including 18 ACC teams; pandas names the duplicate headers `rank` and `rank.1`; use the documented `rank.1` for Barthag rank. ESPN scores are text and start times are UTC. The saved November 14 response contains 3 completed games. These inspections do not yet constitute cleaned tables.

In [5]:
# TODO: the team name crosswalk, with the unmatched names printed before and after

In [6]:
# TODO: games, with the Eastern game date derived from the UTC start time

In [7]:
# TODO: team_games, one row per team per game, with margin and possessions

In [8]:
# TODO: player_games

### 3. Load into SQLite

**TODO:** create the tables with primary keys, load them so that a rerun cannot duplicate rows, and print the row count of every table.

In [9]:
# TODO: schema

In [10]:
# TODO: load, then prove that a second run leaves the counts unchanged

### 4. Questions

Each question gets a SQL query run against your database, a chart if the PDF asks for one, and an answer written in full sentences that includes the number it rests on.

#### Q1. Record versus rating

**Owner:** Nathaniel Garcia

In [11]:
# TODO: SQL

In [12]:
# TODO: chart 1

_Q1 answer._

#### Q2. Strength of the opponents so far

**Owner:** Nathaniel Garcia

In [13]:
# TODO: SQL

In [14]:
# TODO: chart 2

_Q2 answer._

#### Q3. Close games and recent form

**Owner:** Mohamed Hassan

In [15]:
# TODO: SQL, including a window function for the rolling margin

In [16]:
# TODO: chart 3

_Q3 answer._

#### Q4. Pace and points per possession

**Owner:** Mouad Tazka

In [17]:
# TODO: SQL

_Q4 answer._

#### Q5. Who carries each team

**Owner:** Mouad Tazka

In [18]:
# TODO: SQL

_Q5 answer._

#### Q6. Does the rating predict the winner?

**Owner:** Mohamed Hassan

In [19]:
# TODO: SQL that uses the latest snapshot taken before each game

_Q6 answer, with the number of games it covers._

### 5. Decision log

No transformations have been applied yet. The following are planned tasks from the kickoff; record actual affected-row counts here when each step is implemented.

| Planned task | Evidence from the kickoff |
|---|---|
| Match ESPN IDs and source names; validate all 18 ACC teams | Torvik `N.C. State`, `Miami FL`, `Florida St.` correspond to ESPN `NC State`, `Miami`, `Florida State` in the saved team directory. |
| Use `rank.1` as the national Barthag rank | Both raw headers are `rank`; pandas disambiguates them. Both columns agree in this snapshot. |
| Convert scores to numbers | Georgia Tech's `'75'` and UNC Asheville's `'74'` are strings. |
| Derive game dates in Eastern time | All 3 games inspected stay on November 14 after conversion; later evening games may cross the UTC date boundary. |
| Keep finished games and retain venue flags | All 3 inspected games are completed and non-neutral. |
| Reshape to one row per team per game | The 3 inspected games would produce 6 team-game rows. |
| Investigate unmatched opponents before deciding how to handle missing ratings | All 3 opponents on November 14 have Torvik rows. |
| Keep rating snapshot dates separate from game dates | The November 18 snapshot is too late to predict November 14 games for Q6. |
| Split shooting statistics and flatten player box scores | Pending the first summary pulls; no box-score transformation has been performed. |


In [20]:
# TODO

### 6. The published API

**TODO:** the live address, the list of routes, and chart 4, drawn from JSON returned by your own API.

In [21]:
API_URL = 'TODO'

# TODO: call your routes with requests and draw chart 4

### 7. Reflection

**TODO:** answer the five reflection questions from the PDF, with every member contributing.

_reflection here_

### 8. Presentation video

**TODO:** the link to your team's recorded presentation, which runs no more than eight minutes and includes every member. Put the same link in the README.

_video link here_